# Error Analysis: Baseline RT-DETRv2 (baseline-001, epoch 19)
Purpose, split (val), thresholds (score 0.5, IoU 0.5), one class.
Data sources: matches.csv, summary.json, coco_metrics.json, review_tags.csv

# 1. EDA: Quick Look at the Results

### Setup and load data 


In [5]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

EVAL_DIR = ROOT / "runs/baseline-001/eval-epoch19/error-analysis"

with (EVAL_DIR / "summary.json").open(encoding="utf-8") as file:
    summary = json.load(file)

with (EVAL_DIR / "coco_metrics.json").open(encoding="utf-8") as file:
    coco_metrics = json.load(file)

matches = pd.read_csv(EVAL_DIR / "matches.csv")

print(f"Loaded summary, coco_metrics, and matches ({len(matches):,} rows).")


Loaded summary, coco_metrics, and matches (2,252 rows).


In [7]:
summary

{'TP': 2019,
 'FP': 57,
 'FN': 176,
 'precision': 0.9725433526011561,
 'recall': 0.9198177676537586,
 'F1': 0.9454460313743854,
 'score_threshold': 0.5,
 'iou_threshold': 0.5,
 'fp_type_counts': {'duplicate': 3, 'poor_localization': 29, 'background': 25},
 'fn_type_counts': {'matched_elsewhere': 0,
  'low_confidence': 29,
  'not_detected': 147}}

In [8]:
coco_metrics 

{'AP': 0.6564750647692962,
 'AP50': 0.9323106624735741,
 'AP75': 0.7713782018206488,
 'AP_small': 0.41893912853055676,
 'AP_medium': 0.7143128132781181,
 'AP_large': 0.6686455973498184,
 'AR_1': 0.6798633257403188,
 'AR_10': 0.7233257403189066,
 'AR_100': 0.7356719817767654,
 'AR_small': 0.5113110539845759,
 'AR_medium': 0.7819908316961361,
 'AR_large': 0.7935714285714285}

In [12]:
matches.describe()

,image_id,score,pred_x,pred_y,pred_w,pred_h,gt_x,gt_y,gt_w,gt_h,iou,box_area_ratio,best_overlap_score
count,2252.000000,2076.000000,2076.000000,2076.000000,2076.000000,2076.000000,2195.000000,2195.000000,2195.000000,2195.000000,2074.000000,2252.000000,98.000000
mean,1026.310835,0.869161,225.595860,187.575607,123.393792,63.201482,238.364465,194.599544,116.579010,58.493408,0.830136,0.049411,0.093669
std,584.640354,0.051236,142.420981,132.303867,149.638523,79.794620,165.818036,140.043028,143.297778,74.730813,0.130658,0.123150,0.129558
min,0.000000,0.508057,-6.879580,-1.494658,11.256657,7.000526,0.000000,1.000000,3.300000,1.600000,0.000000,0.000012,0.001813
25%,523.750000,0.859655,164.135300,118.724123,71.154011,28.132015,165.000000,120.000000,62.000000,26.000000,0.795727,0.010279,0.008135
50%,1027.500000,0.874491,203.039963,163.316673,93.364563,39.606232,205.000000,165.000000,92.000000,37.000000,0.860557,0.023277,0.022119
75%,1534.250000,0.898601,247.291302,217.176254,111.611404,72.688332,254.000000,223.000000,112.000000,70.000000,0.906295,0.042792,0.140452
max,2047.000000,0.934261,1812.887939,1257.466187,1641.129700,881.758850,1813.000000,1253.000000,1545.820000,846.000000,0.985135,0.989521,0.478098


### Sanity checks
   - cell: TP + FN == 2195, total rows == 2252

In [16]:
tp_count = (matches["type"] == "TP").sum()
fn_count = (matches["type"] == "FN").sum()

print(f"TP: {tp_count:,}")
print(f"FN: {fn_count:,}")
print(f"TP + FN: {tp_count + fn_count:,}")

assert tp_count + fn_count == 2195, "TP + FN must equal the number of GT boxes"
print(f"total images {len(matches["image_id"])}")

TP: 2,019
FN: 176
TP + FN: 2,195
total images 2252


### Error breakdown
   - cell: value_counts of type, fp_type, fn_type

In [18]:
for column in ("type", "fp_type", "fn_type"):
    print(f"{column} counts:")
    print(matches[column].value_counts())
    print()


type counts:
type
TP    2019
FN     176
FP      57
Name: count, dtype: int64

fp_type counts:
fp_type
poor_localization    29
background           25
duplicate             3
Name: count, dtype: int64

fn_type counts:
fn_type
not_detected      147
low_confidence     29
Name: count, dtype: int64



In [27]:
# CHECK by the plate size 
gt = matches[matches.type.isin(['TP','FN'])].copy()
gt['found'] = gt.type == 'TP'
gt['size_bin'] = pd.qcut(gt.box_area_ratio, 5)
gt.groupby('size_bin', observed=True).found.agg(['mean','count'])

,mean,count
size_bin,,
"(-0.0009884, 0.00689]",0.674260,439
"(0.00689, 0.0184]",0.970455,440
"(0.0184, 0.0288]",0.990868,438
"(0.0288, 0.0467]",0.990909,440
"(0.0467, 0.933]",0.972603,438


In [28]:
gt.groupby('fn_type', observed=True).found.agg(['mean','count'])

,mean,count
fn_type,,
low_confidence,0.0,29
not_detected,0.0,147


# 2. Quantitative Analysis

### Recall by plate size
   - cell: qcut on box_area_ratio, recall per bin

### Plate size of misses by fn_type
   - cell: groupby fn_type → box_area_ratio.describe()

## Confidence of FPs and low_confidence FNs (optional)
   - cell: score distribution

# 3. Visual Review

### Image viewer helper
   - function to show one image / a grid of 6


### FN: not_detected (50)

### FN: low_confidence (10)

### FP: all (57)
   - Tagging itself happens in the spreadsheet (review_tags.csv)

# 4. Tag Counts

### Load tagged review_tags.csv

### Counts and percentages per subtype
- cell: `groupby(type, subtype).category.value_counts()`

# 5. Top Failure Modes and Hypotheses

### Failure mode 1: ___
- Evidence (counts, example image) → Hypothesis → Cheap check → Proposed fix

### Failure mode 2: ___
- Evidence (counts, example image) → Hypothesis → Cheap check → Proposed fix

### Failure mode 3: ___
- Evidence (counts, example image) → Hypothesis → Cheap check → Proposed fix

# 6. Numbers for the Report

# 7. Limitations and Next Steps
- Val set used for checkpoint selection
- Subjective tags
- 50-of-147 sample
- No test set